#### Transform Customer Data

1. Remove record with NULL customer_id
2. Remove exact Duplicate Records
3. Remove Duplicate Records based on created_timestamp
4. Cast the columns to correct Data Type
5. Write transformed data to the Silver schema

##### 1. Remove record with NULL customer_id

In [0]:
SELECT * 
    from
gizmobox_catalog_subbu.bronze.v_customers 
where customer_id is NOT null

##### 2. Remove exact Duplicate Records

In [0]:
SELECT DISTINCT * 
    from
gizmobox_catalog_subbu.bronze.v_customers 
where customer_id is NOT null
ORDER BY customer_id

In [0]:
SELECT customer_id,
        max(created_timestamp),
        max(customer_name),
        max(date_of_birth),
        max(email),
        max(member_since)
    FROM gizmobox_catalog_subbu.bronze.v_customers
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
    ORDER BY customer_id


In [0]:
CREATE OR REPLACE TEMPORARY VIEW tv_customers_distinct
 AS
SELECT DISTINCT * 
    from
gizmobox_catalog_subbu.bronze.v_customers 
where customer_id is NOT null
ORDER BY customer_id

In [0]:
select * from tv_customers_distinct

In [0]:
select customer_id,
max(created_timestamp) as max_timestamp
 from tv_customers_distinct
group by customer_id

##### 3. Remove Duplicate Records based on created_timestamp

In [0]:
WITH cte_max AS
(
  select customer_id,
max(created_timestamp) as max_timestamp
 from tv_customers_distinct
group by customer_id
)
SELECT t.*
    FROM tv_customers_distinct t
    join cte_max m
    on t.customer_id = m.customer_id
    and t.created_timestamp = m.max_timestamp


##### 4. Cast the columns to correct Data Type

In [0]:
WITH cte_max AS
(
 select customer_id,
    max(created_timestamp) as max_timestamp
 from tv_customers_distinct
 group by customer_id
)
SELECT cast(t.created_timestamp as timestamp) as created_timestamp,
        t.customer_id,
        t.customer_name,
        cast(t.date_of_birth as date) as date_of_birth,
        t.email,
        cast(t.member_since as date) as member_since,
        t.telephone
    FROM tv_customers_distinct t
    join cte_max m
    on t.customer_id = m.customer_id
    and t.created_timestamp = m.max_timestamp

##### 5. Write transformed data to the Silver schema (Write data to Delta table)

In [0]:
CREATE TABLE gizmobox_catalog_subbu.silver.customers
AS
WITH cte_max AS
(
 select customer_id,
    max(created_timestamp) as max_timestamp
 from tv_customers_distinct
 group by customer_id
)
SELECT cast(t.created_timestamp as timestamp) as created_timestamp,
        t.customer_id,
        t.customer_name,
        cast(t.date_of_birth as date) as date_of_birth,
        t.email,
        cast(t.member_since as date) as member_since,
        t.telephone
    FROM tv_customers_distinct t
    join cte_max m
    on t.customer_id = m.customer_id
    and t.created_timestamp = m.max_timestamp

In [0]:
select * from gizmobox_catalog_subbu.silver.customers

In [0]:
DESCRIBE EXTENDED gizmobox_catalog_subbu.silver.customers